<a href="https://colab.research.google.com/github/nestarShaxzod/IFRS9-Expected-Credit-Loss-Automation/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22IFRS_9_ECL_automatization_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Автоматизация расчета ожидаемых кредитных убытков (IFRS 9 ECL)***
------------------------------------------------------------------------
**Цель:** демонстрация автоматизации расчета ожидаемых кредитных убытков (Expected Credit Loss, ECL) в соответствии с требованиями IFRS 9 с использованием Python (Pandas) и DuckDB SQL.


Данные и пути к источникам обезличены для публичной публикации.




In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import duckdb
pd.set_option('display.float_format', '{:,.2f}'.format)

# ***Пути к входным и выходным данным***

In [ ]:
path_input = r'./data/input'
path_output = r'./data/output'

# ***Функции преобразования типов данных***

In [ ]:
#______________________________________________Функция для изменения на тип "Дата"______________________________________________
def def_date_type(df, col):
    df[col] = pd.to_datetime(df[col], dayfirst=True, format='%d.%m.%Y', errors = 'coerce')
    return df

#______________________________________________Функция для изменения типов чисел________________________________________________
def def_numb_type(df, col):
    df[col] = df[col].str.replace(',', '.', regex = True)
    df[col] = df[col].str.replace(r'\s+', '', regex = True)
    df[col] = pd.to_numeric(df[col], errors = 'coerce')
    return df

# ***Загрузка и объединение исходных файлов***

In [ ]:
file_path = glob.glob(os.path.join(path_input, '*.txt'))
all_frame = []

for f in file_path:

    line_skip_row = 0
    with open(f, 'r', encoding = 'utf-8') as f_open:                             #Для корректного чтения необходимо удалить строки до строки, который содержит слово "Период"
        for numb, row_content in enumerate(f_open):
          if 'Период' in row_content:
            line_skip_row = numb
            break

    df = pd.read_csv(f, sep = '\t', skiprows = line_skip_row, encoding = 'utf-8', usecols=[0,2,3,4,5,7,8]) #Оставляем только нужные столбцы
    all_frame.append(df)

concat_df = pd.concat(all_frame, ignore_index=True)

concat_df = def_date_type(concat_df, 'Период')
concat_df = def_numb_type(concat_df, 'Unnamed: 5')
concat_df = def_numb_type(concat_df, 'Unnamed: 8')

concat_df = concat_df.rename(columns={'Период': 'Date', 'Аналитика Дт': 'Аналитика_Дт', 'Аналитика Кт': 'Аналитика_Кт', 'Unnamed: 5': 'Сумма_Дт', 'Unnamed: 8': 'Сумма_Кт'})
concat_df.head(7)

pd.concat([concat_df.head(5), concat_df.tail(5)])                                # Для демонстрации полученного результата

# ***Обработка и трансформация данных с использованием DuckDB SQL***

In [ ]:
duckdb.register("duck_ecl_calcul", concat_df)

query_df = """
      SELECT
        *,
        CASE
            WHEN "Дебет" LIKE '40%' THEN STRING_SPLIT(REPLACE(Аналитика_Дт, CHR(13), ''), CHR(10))[1]
            ELSE STRING_SPLIT(REPLACE(Аналитика_Кт, CHR(13), ''), CHR(10))[1]
        END AS "Counterparty",                                                  -- Выделяем Контрагента из анатилитики выгрузки
        CASE
            WHEN "Дебет" LIKE '40%' THEN STRING_SPLIT(REPLACE(Аналитика_Дт, CHR(13), ''), CHR(10))[2]
            ELSE STRING_SPLIT(REPLACE(Аналитика_Кт, CHR(13), ''), CHR(10))[2]
        END AS "Contract",                                                      -- Выделяем Контракта из анатилитики выгрузки
        CASE
            WHEN "Дебет" LIKE '40%' THEN "Сумма_Дт"
            ELSE 0
        END AS "Inflow",                                                        -- Увеличение долга оформляем как поступление
        CASE
            WHEN "Кредит" LIKE '40%' THEN "Сумма_Кт" *-1
            ELSE 0
        END AS "Payment"                                                        -- Закрытие долга со строны контрагентв оформляем как выбытие с знаком минус
      FROM duck_ecl_calcul
      WHERE "Date" is not null and "Date" <= '2025-12-31'
      ORDER BY "Date", "Counterparty", "Contract"                               -- Фильтрации по дате внутри контрагента в разрезе контрактов
"""

duck_df = duckdb.query(query_df).df()



pd.concat([duck_df.head(5),duck_df.tail(5)])                         # Для демонстрации полученного результата

# ***Дополнительная обработка данных в DuckDB***

In [ ]:
query_df2 = """
      WITH T1 AS (
          SELECT
            "Date",
            "Counterparty",
            "Contract",
            "Inflow",
            "Payment",
            SUM("Inflow") OVER(PARTITION BY "Counterparty", "Contract" ORDER BY "Date") AS "Cum_Inflow", -- Считаем накопленное поступление
            SUM("Payment") OVER(PARTITION BY "Counterparty", "Contract") AS "Total_Payment",             -- Считаем накопленное погашение по контракту
        FROM duck_df
      ),
          T2 AS (
            SELECT
              *,
              CASE
                WHEN "Cum_Inflow" < ABS("Total_Payment") THEN 0
                ELSE "Cum_Inflow" + "Total_Payment"
              END AS "Cum_Unpaid_amount"
            FROM T1
          )
                SELECT
                  *,
                  CASE
                      WHEN COALESCE(LAG("Cum_Unpaid_amount") OVER(PARTITION BY "Counterparty", "Contract" ORDER BY "Date"),0) = "Cum_Unpaid_amount" THEN 0
                      ELSE "Cum_Unpaid_amount" - LAG("Cum_Unpaid_amount") OVER(PARTITION BY "Counterparty", "Contract" ORDER BY "Date")
                  END AS "Unpaid_amount"
                FROM T2

"""

duck_source_info= duckdb.query(query_df2).df()

pd.concat([duck_source_info.head(5), duck_source_info.tail(5)])           # Для демонстрации полученного результата

# ***Расчет Aging Analysis и формирование возрастных бакетов***

In [ ]:
date_input = '2025-12-31'

query_df3 = f"""

          SELECT
              Date,
              "Counterparty",
              "Contract",
              "Unpaid_amount",
              DATEDIFF('day', "Date", '{date_input}') AS "Days_to_date",                -- Находим разницу между датой отчета и датой возникновения задолженности
              CASE
                  WHEN DATEDIFF('day', "Date", '{date_input}') <= 30 THEN '0-30'
                  WHEN DATEDIFF('day', "Date", '{date_input}') <= 60 THEN '31-60'
                  WHEN DATEDIFF('day', "Date", '{date_input}') <= 90 THEN '61-90'
                  WHEN DATEDIFF('day', "Date", '{date_input}') <= 120 THEN '91-120'
                  ELSE '120 и более'                                                    -- Дни просрочек разносим по Баккетам (Корзинке задолженности)
              END AS "Aging_Bucket"
          FROM duck_source_info
          WHERE "Unpaid_amount" > 1                                                     -- Фильтрируем нужные данные
          ORDER BY "Date", "Counterparty", "Contract"

"""

duck_df_31_12_2025 = duckdb.sql(query_df3).df()

pd.concat([duck_df_31_12_2025.head(5),duck_df_31_12_2025.tail(5)])                  # Для демонстрации полученного результата

# ***Расчет показателей PD и LGD***

In [ ]:
# Применяем заранее определенные значения PD и LGD для упрощения и повышения читаемости кода.

df_pd = {'0-30': 0.05,
          '31-60': 0.15,
          '61-90': 0.25,
          '91-120': 0.35,
          '120 и более': 1
          }

df_lgd = 0.45

# ***Расчет ожидаемых кредитных убытков (Expected Credit Loss, ECL)***

In [ ]:
df_ecl = duck_df_31_12_2025.copy()

df_ecl['PD'] = df_ecl['Aging_Bucket'].map(df_pd)                                # Делаем Меппинг баккетов и PD
df_ecl['LGD'] = df_lgd
df_ecl['ECL'] = df_ecl['Unpaid_amount'] * df_ecl['PD'] * df_ecl['LGD']          # Выводим сумму ожидаемого кредитного убытка по станларту МСФО IFRS 9


pd.concat([df_ecl.head(5),df_ecl.tail(5)])                                      # Для демонстрации полученного результата

# ***Сохранение результатов обработки***

In [ ]:
path_save = os.path.join(path_output, 'ECL.xlsx')

with pd.ExcelWriter(path_save) as writer:
  df_ecl.to_excel(writer, sheet_name='ECL', index=False)
